In [6]:
import pandas as pd
import numpy as np 
import requests
import os
print('Loaded libraries Loaded')

Loaded libraries Loaded


In [ ]:
print('Downloading OpenPhish Feed')
response = requests.get('https://openphish.com/feed.txt', timeout=10)
phishing_urls = response.text.strip().split('\n')

print(f'Total Phishing URLs downloaded : {len(phishing_urls)}')
print(f'Sample URL:')
for url in phishing_urls[:5]:
    print(f"  {url}")

Total Phishing URLs downloaded : 300
Sample URL:
  http://sp22ct-zarur-biz-loru-kemu.pages.dev/
  https://sp33ct-fenaru-biz-kelovi-maneto.pages.dev/
  https://buzqari-mpt-felvuno-p4d9ka86.pages.dev/
  http://mt1xv-55n-xil2-hbkcy-30-09-2026-hh.pages.dev/
  http://sp22ct-vemak-biz-keli-pazu.pages.dev/


In [4]:
import requests
import pandas as pd
import numpy as np
import os
import time

os.makedirs("../data/raw", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../data/batches", exist_ok=True)
print("Folders ready")

Folders ready


In [6]:
phishing_urls = set()

try:
    r = requests.get('https://openphish.com/feed.txt', timeout=30)
    urls = [u.strip() for u in r.text.strip().split('\n') if u.strip().startswith('http')]
    phishing_urls.update(urls)
    print(f'OpenPhish: {len(urls)} URLs')
except Exception as e:
    print(f'OpenPhish failed : {e}')

try:
    r = requests.get(
        "http://data.phishtank.com/data/online-valid.csv",
        timeout=60,
        headers={"User-Agent": "phishtank/youremail@example.com"}
    )
    pt_df = pd.read_csv(__import__('io').StringIO(r.text))
    pt_urls = pt_df["url"].dropna().tolist()
    phishing_urls.update(pt_urls)
    print(f"PhishTank: {len(pt_urls)} URLs")
except Exception as e:
    print(f"PhishTank failed: {e}")

try:
    r = requests.get("https://urlhaus.abuse.ch/downloads/csv_recent/", timeout=60)
    lines = r.text.split("\n")
    data_lines = [l for l in lines if not l.startswith("#") and l.strip()]
    uh_df = pd.read_csv(__import__('io').StringIO("\n".join(data_lines)), 
                         on_bad_lines='skip')
    if 'url' in uh_df.columns:
        uh_urls = uh_df['url'].dropna().tolist()
        phishing_urls.update(uh_urls)
        print(f"URLhaus: {len(uh_urls)} URLs")
except Exception as e:
    print(f"URLhaus failed: {e}")

try:
    r = requests.get(
        "https://raw.githubusercontent.com/mitchellkrogza/Phishing.Database/master/phishing-links-ACTIVE.txt",
        timeout=60
    )
    gh_urls = [u.strip() for u in r.text.split("\n") if u.strip().startswith("http")]
    phishing_urls.update(gh_urls)
    print(f"Phishing.Database GitHub: {len(gh_urls)} URLs")
except Exception as e:
    print(f"GitHub Phishing DB failed: {e}")

phishing_urls = list(phishing_urls)
print(f"\nTotal unique phishing URLs: {len(phishing_urls)}")

with open('../data/raw/openphish_urls.txt', 'w') as f:
    f.write('\n'.join(phishing_urls))
print('Saved')

OpenPhish: 300 URLs
PhishTank failed: 'url'
Phishing.Database GitHub: 789052 URLs

Total unique phishing URLs: 789352
Saved


In [10]:
print("Downloading Majestic Million (Alexa replacement)...")
majestic_url = "https://downloads.majestic.com/majestic_million.csv"
alexa_df = pd.read_csv(majestic_url)
alexa_df = alexa_df[["GlobalRank", "Domain"]].rename(columns={"GlobalRank": "rank", "Domain": "domain"})

print(f"Total domains: {len(alexa_df)}")
alexa_df.head()

Total domains: 1000000


,rank,domain
0,1,google.com
1,2,facebook.com
2,3,youtube.com
3,4,instagram.com
4,5,twitter.com


In [11]:
SAMPLE_SIZE = 15000  

phishing_sample = pd.DataFrame({
    "url": np.random.choice(phishing_urls, size=min(SAMPLE_SIZE, len(phishing_urls)), replace=False),
    "label": 1
})

top_domains = alexa_df.head(SAMPLE_SIZE * 2)["domain"].tolist()  
legit_urls = ["https://" + d for d in top_domains]
legit_sample = pd.DataFrame({
    "url": legit_urls[:SAMPLE_SIZE],
    "label": 0
})

df = pd.concat([phishing_sample, legit_sample], ignore_index=True)
df = df.drop_duplicates(subset="url").sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Total URLs: {len(df)}")
print(df["label"].value_counts())
df.to_csv("../data/raw/urls_with_labels.csv", index=False)
print("Saved to data/raw/urls_with_labels.csv")
df.head()

Total URLs: 30000
label
1    15000
0    15000
Name: count, dtype: int64
Saved to data/raw/urls_with_labels.csv


,url,label
0,http://yaho-world-wandering-limit-8b7e.yorzeko...,1
1,https://westlaw.com,0
2,https://google.mg,0
3,https://charlotte.edu,0
4,https://mesesemondre.web.app,1
